In [ ]:

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sqlalchemy import create_engine
from clickhouse_driver import Client
import gc

client = Client(
    host="192.168.1.131",
    port=9000,
    user='MUNA1',
    password='Buyandelger2025**',
    settings={
        "max_execution_time": 1800,
        "receive_timeout": 600,
        "send_timeout": 600,
        "connect_timeout": 60,
        "max_memory_usage": 8 * 1024**3,
        "max_server_memory_usage": 46 * 1024**3,
        "max_threads": 8,
        "max_bytes_before_external_sort": 4 * 1024**3,
        "max_bytes_before_external_group_by": 4 * 1024**3,
        "external_sort_use_cache_for_reads": 1,
        "max_block_size": 65536,
        "preferred_block_size_bytes": 1 * 1024**2,
        "optimize_on_insert": 1,
        "async_insert": 0,
        "insert_deduplicate": 1,
        "max_insert_block_size": 262144,
        "use_uncompressed_cache": 0,
        "log_queries": 1,
        "log_queries_min_type": "EXCEPTION_WHILE_PROCESSING"
    }
)

# =========================================
# 1. EMPLOYEE + CIF MAPPING
# =========================================

query_erp = """
SELECT DISTINCT
    E.EMPLOYEE_ID,
    E.FIRST_NAME,
    E.EMPLOYEE_CODE,
    E.STATUS_NAME,
    E.SOL_ID,
    A.ORGKEY AS CIF_ID
FROM ERP.ERP_EMPLOYEE AS E
LEFT JOIN FINACLE.ACCOUNTS AS A
    ON E.STATE_REG_NUMBER = A.STRFIELD12
LEFT JOIN ERP.ERP_EMPLOYEE_KEY AS EK
    ON E.EMPLOYEE_ID = EK.EMPLOYEE_ID
WHERE
    EK.IS_ACTIVE = 1
    AND E.CURRENT_STATUS_ID IN (1, 3)
    AND A.ORGKEY IS NOT NULL
"""
data, columns = client.execute(query_erp, with_column_types=True)
column_names = [col[0] for col in columns]

emp_df = pd.DataFrame(data, columns=column_names)
emp_df = emp_df.dropna(subset=["CIF_ID"])
emp_df["CIF_ID"] = emp_df["CIF_ID"].astype(str).str.strip()
emp_df = emp_df.drop_duplicates(subset=["CIF_ID"])

emp_cif_list = emp_df["CIF_ID"].tolist()
emp_cif_str = ", ".join([f"'{c}'" for c in emp_cif_list])

print(f"Нийт ажилтны тоо: {len(emp_df)}")

# =========================================
# 2. TRANSACTION DATA — 12 сарын түүх татах
#    (baseline тооцооны тулд 2025-01-01-аас)
# =========================================

HISTORY_START = "2025-01-01"
SCORE_START   = "2026-01-01"

query_tx = f"""
SELECT
    GA.CIF_ID,
    T3.H_ENTRY_DATE,
    T3.H_TRAN_AMT * T3.B_ACCT_RATE AS H_TRAN_AMT_MNT
FROM FINACLE.HTD_ATD T3
JOIN FINACLE.GAM_ACCOUNTS GA
    ON GA.ACID = T3.H_ACID
WHERE T3.H_PART_TRAN_TYPE = 'D'
  AND T3.H_TRAN_AMT > 0
  AND T3.H_TRAN_DATE >= toDate('{HISTORY_START}')
  AND GA.CIF_ID IN ({emp_cif_str})
"""
data, columns = client.execute(query_tx, with_column_types=True)
column_names = [col[0] for col in columns]
df = pd.DataFrame(data, columns=column_names)

df["H_ENTRY_DATE"] = pd.to_datetime(df["H_ENTRY_DATE"])
df["date"]         = df["H_ENTRY_DATE"].dt.date
df["year_month"]   = df["H_ENTRY_DATE"].dt.to_period("M")
df["is_weekend"]   = df["H_ENTRY_DATE"].dt.dayofweek >= 5
df["CIF_ID"]       = df["CIF_ID"].astype(str).str.strip()

df = df.sort_values(["CIF_ID", "H_ENTRY_DATE"])
print(f"Нийт гүйлгээний мөр: {len(df):,}")


In [ ]:

# =========================================
# 3. DAILY AGGREGATION
# =========================================

daily = (
    df.groupby(["CIF_ID", "date", "year_month", "is_weekend"])
      .agg(
          daily_sum=("H_TRAN_AMT_MNT", "sum"),
          daily_cnt=("H_TRAN_AMT_MNT", "count"),
          daily_max=("H_TRAN_AMT_MNT", "max")
      )
      .reset_index()
)

# =========================================
# 4. MONTHLY BASELINE — зөвхөн HISTORY-аас
# =========================================

monthly = (
    df.groupby(["CIF_ID", "year_month"])
      .agg(
          monthly_sum=("H_TRAN_AMT_MNT", "sum"),
          monthly_cnt=("H_TRAN_AMT_MNT", "count")
      )
      .reset_index()
      .sort_values(["CIF_ID", "year_month"])
)

# Time-aware expanding baseline (shift(1) = өнгөрсөн мэдээлэл л хэрэглэнэ)
monthly["hist_avg_monthly"] = (
    monthly.groupby("CIF_ID")["monthly_sum"]
           .expanding().mean().shift(1)
           .reset_index(level=0, drop=True)
)
monthly["hist_std_monthly"] = (
    monthly.groupby("CIF_ID")["monthly_sum"]
           .expanding().std().shift(1)
           .reset_index(level=0, drop=True)
)
monthly["hist_months"] = (
    monthly.groupby("CIF_ID")["monthly_sum"]
           .expanding().count().shift(1)
           .reset_index(level=0, drop=True)
)
monthly["hist_avg_cnt"] = (
    monthly.groupby("CIF_ID")["monthly_cnt"]
           .expanding().mean().shift(1)
           .reset_index(level=0, drop=True)
)

# 75th и 95th percentile-ийн rolling (харьцуулалтыг олон хэмжүүрт тулгуурлах)
def rolling_percentile(series, pct):
    result = series.shift(1).expanding().quantile(pct / 100)
    return result

monthly["hist_p75"] = (
    monthly.groupby("CIF_ID")["monthly_sum"]
           .apply(lambda s: s.shift(1).expanding().quantile(0.75))
           .reset_index(level=0, drop=True)
)
monthly["hist_p95"] = (
    monthly.groupby("CIF_ID")["monthly_sum"]
           .apply(lambda s: s.shift(1).expanding().quantile(0.95))
           .reset_index(level=0, drop=True)
)

# =========================================
# 5. SCORE ХУГАЦААНД FILTER ХИЙХ
#    (Baseline тооцоолсны дараа — 2026-01-01+)
# =========================================

score_period = pd.Period(SCORE_START[:7], freq="M")
daily_score  = daily[daily["year_month"] >= score_period].copy()

scoring = daily_score.merge(
    monthly[["CIF_ID", "year_month",
             "hist_avg_monthly", "hist_std_monthly",
             "hist_months", "hist_avg_cnt",
             "hist_p75", "hist_p95"]],
    on=["CIF_ID", "year_month"],
    how="left"
)

# =========================================
# 6. FEATURE ТООЦООЛОЛ
# =========================================

# Өдрийн хэвийн дундаж: сарын дундаж / 22 ажлын өдөр (7-аас бол 30)
scoring["daily_avg_base"] = scoring.apply(
    lambda r: r["hist_avg_monthly"] / 22
              if not r["is_weekend"]
              else r["hist_avg_monthly"] / 8,
    axis=1
)
scoring["daily_std_base"] = scoring.apply(
    lambda r: r["hist_std_monthly"] / 22
              if not r["is_weekend"]
              else r["hist_std_monthly"] / 8,
    axis=1
)

scoring["relative_spike"] = scoring["daily_sum"] / scoring["daily_avg_base"].replace(0, np.nan)
scoring["z_score"]        = (scoring["daily_sum"] - scoring["daily_avg_base"]) / scoring["daily_std_base"].replace(0, np.nan)
scoring["pct_of_monthly"] = scoring["daily_sum"] / scoring["hist_avg_monthly"].replace(0, np.nan)

# =========================================
# 7. ADAPTIVE RULE-BASED SCORING
# =========================================

MIN_HISTORY_MONTHS = 3  # 2026 жилийн хувьд realistic болгосон (хуучин 6 байсан)

def detect_anomaly(r):
    # Хангалттай түүхгүй бол → зөвхөн абсолют threshold ашиглана
    has_history = (
        not pd.isna(r["hist_avg_monthly"]) and
        not pd.isna(r["hist_std_monthly"]) and
        r["hist_months"] >= MIN_HISTORY_MONTHS
    )

    score   = 0.0
    reasons = []

    # --- A. СТАТИСТИК ДҮРМҮҮД (түүх байвал) ---
    if has_history:
        daily_avg = r["daily_avg_base"]
        daily_std = r["daily_std_base"]

        # Spike: өдрийн дүн хэвийн өдрийн дундажаас 5 дахин их
        if r["relative_spike"] >= 5:
            score += 2.5
            reasons.append(f"Spike {r['relative_spike']:.1f}x (хэвийн дундажаас)")

        elif r["relative_spike"] >= 3:
            score += 1.5
            reasons.append(f"Spike {r['relative_spike']:.1f}x (хэвийн дундажаас)")

        # Z-score: статистик хазайлт
        if r["z_score"] >= 5:
            score += 2.0
            reasons.append(f"Z-score={r['z_score']:.1f} (маш өндөр)")
        elif r["z_score"] >= 3:
            score += 1.0
            reasons.append(f"Z-score={r['z_score']:.1f}")

        # Сарын 75% нормоос хэт давсан (нэг өдрийн гүйлгээ)
        if not pd.isna(r["hist_p75"]) and r["daily_sum"] > r["hist_p75"]:
            score += 0.5
            reasons.append("Сарын P75-аас давсан өдөр")

        # Гүйлгээний давтамж: хэт олон гүйлгээ нэг өдөрт
        if not pd.isna(r["hist_avg_cnt"]) and r["hist_avg_cnt"] > 0:
            cnt_ratio = r["daily_cnt"] / (r["hist_avg_cnt"] / 22)
            if cnt_ratio >= 5:
                score += 1.5
                reasons.append(f"Гүйлгээний тоо {cnt_ratio:.1f}x (хэт олон)")
            elif cnt_ratio >= 3:
                score += 0.8
                reasons.append(f"Гүйлгээний тоо {cnt_ratio:.1f}x")

    # --- B. АБСОЛЮТ THRESHOLD (түүх хэрэггүй) ---
    # 50 сая ₮-аас дээш нэг өдрийн нийт зарлага
    if r["daily_sum"] >= 50_000_000:
        score += 2.0
        reasons.append(f"Нэг өдрийн зарлага {r['daily_sum']/1e6:.1f}M ₮ (>50M)")
    elif r["daily_sum"] >= 20_000_000:
        score += 1.0
        reasons.append(f"Нэг өдрийн зарлага {r['daily_sum']/1e6:.1f}M ₮ (>20M)")

    # Нэг гүйлгээ 10 сая ₮-аас дээш
    if r["daily_max"] >= 10_000_000:
        score += 1.0
        reasons.append(f"Нэг гүйлгээ {r['daily_max']/1e6:.1f}M ₮ (>10M)")

    # Амралтын өдрийн гүйлгээ
    if r["is_weekend"] and r["daily_sum"] >= 5_000_000:
        score += 0.5
        reasons.append("Амралтын өдөр >5M ₮ зарлага")

    is_alert = score >= 2.5

    if not reasons:
        reason_str = "Хэвийн" if has_history else "Түүх хангалтгүй (<3 сар)"
    else:
        reason_str = "; ".join(reasons)

    return is_alert, round(score, 2), reason_str

results = scoring.apply(detect_anomaly, axis=1, result_type="expand")
scoring["is_alert"]   = results[0]
scoring["risk_score"] = results[1]
scoring["reason"]     = results[2]

# Risk level ангилал
def risk_level(score):
    if score >= 5:   return "МАШ ӨНДӨР"
    if score >= 3.5: return "ӨНДӨР"
    if score >= 2.5: return "ДУНД"
    if score >= 1.0: return "БАГА"
    return "ХЭВИЙН"

scoring["risk_level"] = scoring["risk_score"].apply(risk_level)

alert_cnt = scoring["is_alert"].sum()
print(f"Нийт alert: {alert_cnt:,} / {len(scoring):,} өдрийн бичлэг")
print(scoring[scoring["is_alert"]]["risk_level"].value_counts())


In [ ]:

# =========================================
# 8. АЖИЛТНЫ МЭДЭЭЛЭЛ НЭМЭХ + ЭЦСИЙН ХҮСНЭГТ
# =========================================

scoring["score_date"] = pd.to_datetime(scoring["date"])
scoring["insert_ts"]  = datetime.now()

final_scoring = scoring.merge(
    emp_df[["CIF_ID", "FIRST_NAME", "EMPLOYEE_CODE", "SOL_ID", "STATUS_NAME"]],
    on="CIF_ID",
    how="left"
)

final_scoring = final_scoring[[
    "CIF_ID",
    "EMPLOYEE_CODE",
    "FIRST_NAME",
    "SOL_ID",
    "STATUS_NAME",
    "score_date",
    "is_weekend",
    "daily_sum",
    "daily_cnt",
    "daily_max",
    "hist_avg_monthly",
    "hist_std_monthly",
    "hist_months",
    "relative_spike",
    "z_score",
    "pct_of_monthly",
    "risk_score",
    "risk_level",
    "is_alert",
    "reason",
    "insert_ts"
]]

final_scoring.columns = [
    "CIF_ID", "EMPLOYEE_CODE", "EMPLOYEE_NAME", "BRANCH_ID", "STATUS",
    "SCORE_DATE", "IS_WEEKEND",
    "DAILY_SUM", "DAILY_CNT", "DAILY_MAX",
    "HIST_AVG_MONTHLY", "HIST_STD_MONTHLY", "HIST_MONTHS",
    "RELATIVE_SPIKE", "Z_SCORE", "PCT_OF_MONTHLY",
    "RISK_SCORE", "RISK_LEVEL", "IS_ALERT",
    "REASON", "INSERT_TS"
]

print(f"Эцсийн мөрийн тоо: {len(final_scoring):,}")
final_scoring[final_scoring["IS_ALERT"]].sort_values("RISK_SCORE", ascending=False).head(20)


In [ ]:

# =========================================
# 9. ХУРААНГУЙ СТАТИСТИК
# =========================================

summary = (
    final_scoring[final_scoring["IS_ALERT"]]
    .groupby(["EMPLOYEE_CODE", "EMPLOYEE_NAME", "BRANCH_ID", "RISK_LEVEL"])
    .agg(
        alert_days   =("SCORE_DATE", "count"),
        total_amount =("DAILY_SUM", "sum"),
        max_day_amt  =("DAILY_SUM", "max"),
        avg_risk     =("RISK_SCORE", "mean"),
        max_risk     =("RISK_SCORE", "max"),
        last_alert   =("SCORE_DATE", "max")
    )
    .reset_index()
    .sort_values("max_risk", ascending=False)
)

summary["total_amount_M"] = (summary["total_amount"] / 1e6).round(2)
summary["max_day_amt_M"]  = (summary["max_day_amt"]  / 1e6).round(2)

print("=== ДОХИОЛЛЫН ХУРААНГУЙ ===")
summary


In [ ]:

final_scoring


In [ ]:

# =========================================
# 10. EXCEL EXPORT — 2 sheet
# =========================================

output_path = r"F:\employee_risk_scoring.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    final_scoring.to_excel(writer, sheet_name="Бүх бичлэг", index=False)
    final_scoring[final_scoring["IS_ALERT"]].sort_values(
        "RISK_SCORE", ascending=False
    ).to_excel(writer, sheet_name="Дохиоллууд", index=False)
    summary.to_excel(writer, sheet_name="Хураангуй", index=False)

print(f"Хадгалагдлаа: {output_path}")
